# torch.cond with XNNPACK Backend

This notebook demonstrates exporting a model that uses `torch.cond` (a higher-order op for conditional branching) with linear layers in each branch, lowering it to the XNNPACK backend, and validating that the ExecuTorch output matches eager PyTorch execution.

Prerequisites: install ExecuTorch with `./install_executorch.sh`.

In [1]:
import torch
from torch import nn


class CondLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_true = nn.Linear(4, 4)
        self.linear_false = nn.Linear(4, 4)

    def true_fn(self, x):
        return self.linear_true(x)

    def false_fn(self, x):
        return self.linear_false(x)

    def forward(self, pred, x):
        return torch.cond(pred, self.true_fn, self.false_fn, [x])


model = CondLinearModel().eval()
example_inputs = (torch.tensor(True), torch.randn(4))
print(model(*example_inputs))

tensor([-0.5125, -0.2433,  0.8868, -0.4800], grad_fn=<CondAutogradOpBackward>)


In [2]:
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

exported = torch.export.export(model, example_inputs)

et_program = to_edge_transform_and_lower(
    exported,
    partitioner=[XnnpackPartitioner()],
).to_executorch()

pte_path = "cond_model.pte"
with open(pte_path, "wb") as f:
    f.write(et_program.buffer)

print(f"Saved to {pte_path}")

W0305 17:31:14.038000 73481 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/gjcomer/miniconda/envs/executorch/lib/python3.12/site-packages/torch/_higher_order_ops/cond.py:221: UserWarning: You are calling torch.compile inside torch.export region. To capture an useful graph, we will implicitly switch to torch.compile(backend=eager)
  return torch.compile(_cond_op_wrapper, backend=backend, fullgraph=True)(
/Users/gjcomer/miniconda/envs/executorch/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/gjcomer/miniconda/envs/executorch/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/gjcomer/minicond

Saved to cond_model.pte


/Users/gjcomer/miniconda/envs/executorch/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [3]:
from executorch.runtime import Runtime

runtime = Runtime.get()
program = runtime.load_program(pte_path)
method = program.load_method("forward")

# Test with pred=True (should use linear_true)
x = torch.randn(4)
et_output_true = method.execute([torch.tensor(True), x])[0]
eager_output_true = model(torch.tensor(True), x)
match_true = torch.allclose(et_output_true, eager_output_true, rtol=1e-3, atol=1e-5)
print(f"pred=True  match: {match_true}")

# Test with pred=False (should use linear_false)
et_output_false = method.execute([torch.tensor(False), x])[0]
eager_output_false = model(torch.tensor(False), x)
match_false = torch.allclose(et_output_false, eager_output_false, rtol=1e-3, atol=1e-5)
print(f"pred=False match: {match_false}")

# Sanity check: the two branches produce different results
print(f"Branches differ: {not torch.allclose(et_output_true, et_output_false)}")

[program.cpp:163] InternalConsistency verification requested but not available
[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version


pred=True  match: True
pred=False match: True
Branches differ: True


In [4]:
import os

os.remove(pte_path)
print("Cleaned up")

Cleaned up


In [8]:
et_program.dump_executorch_program(True)

Program(
  version=0, 
  execution_plan=[
    ExecutionPlan(
      name=forward, 
      container_meta_type=ContainerMetadata(encoded_inp_str=[1, {"type": "builtins.tuple", "context": "null", "children_spec": [{"type": "builtins.tuple", "context": "null", "children_spec": [{"type": null, "context": null, "children_spec": []}, {"type": null, "context": null, "children_spec": []}]}, {"type": "builtins.dict", "context": "[]", "children_spec": []}]}], encoded_out_str=[1, {"type": null, "context": null, "children_spec": []}]), 
      values=[
        EValue(
          val=Tensor(
            scalar_type=6, 
            storage_offset=0, 
            sizes=[4, 4], 
            dim_order=[0, 1], 
            requires_grad=True, 
            layout=0, 
            data_buffer_idx=1, 
            allocation_info=None, 
            shape_dynamism=0, 
            extra_tensor_info=None
          )
        )(index=0),
        EValue(
          val=Tensor(
            scalar_type=6, 
            st